# 양자컴퓨팅 부트캠프 Day 3: 양자 간섭과 2-qubit Grover 탐색 알고리즘 핸즈온

본 노트북은 **12시간 양자컴퓨팅 부트캠프의 3일차 3교시 실습 교재**입니다.
고전 $O(N)$ 검색의 한계를 넘어 파동 간섭을 통해 정답을 $O(\sqrt{N})$으로 찾아내는 Grover 알고리즘을 NumPy와 Qiskit으로 직접 구현합니다.

---
### 오늘 실습의 5대 학습 목표
1. **Grover 메커니즘 수치 검증**: Phase Oracle과 Diffusion 연산자를 순수 NumPy 행렬곱으로 구현하여 정답 진폭 증폭 확인
2. **Qiskit 회로 조립**: Hadamard 기저 생성 및 Little-Endian(문자열 순서 'q1 q0') 표기 규칙 체득
3. **Phase Oracle 게이트 설계**: 타겟에 따라 $X$ 게이트와 $CZ$ 게이트를 조합하여 부호 반전 오라클 구성
4. **Diffusion 연산자 회로화**: $H^{\otimes 2} X^{\otimes 2} CZ X^{\otimes 2} H^{\otimes 2}$ 패턴을 통한 평균 반전 구현
5. **전수 검증 및 Shots 분석**: 4가지 모든 타겟에 대해 단 1회 질의만으로 100% 검출되는 양자 가속 확인


In [ ]:
from __future__ import annotations

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

np.set_printoptions(precision=4, suppress=True)
print('Qiskit & NumPy ready for Grover search')


## 1. NumPy를 이용한 진폭 증폭(Grover Step) 수치 검증

2큐비트 탐색 공간($N = 2^2 = 4$)에서 타겟이 $|11\rangle$인 경우:
1. **초기 상태 $|s\rangle$**: 모든 서랍에 균등한 진폭 $0.5$ 배분 ($P=0.25$)
2. **Phase Oracle ($O_\omega = I - 2|\omega\rangle\langle\omega|$)**: 정답 상태 $|11\rangle$의 진폭만 $-0.5$로 뒤집음
3. **Diffusion Operator ($D = 2|s\rangle\langle s| - I$)**: 전체 진폭을 평균값에 대해 대칭 반전


In [ ]:
# [실습 과제 1] NumPy를 이용한 Phase Oracle 및 Diffusion 행렬 구현
def numpy_grover_step(target: str = '11') -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # 1. 초기 균등 중첩 |s> = 1/2 [1, 1, 1, 1]^T
    s_state = 0.5 * np.ones((4, 1), dtype=complex)

    # 2. Phase Oracle 행렬: 정답 인덱스의 대각 원소만 -1로 반전
    target_idx = {'00': 0, '01': 1, '10': 2, '11': 3}[target]
    oracle_matrix = None  # TODO: np.eye(4, dtype=complex) 생성 후 target_idx 원소를 -1.0으로 수정하세요
    after_oracle = oracle_matrix @ s_state if oracle_matrix is not None else None

    # 3. Diffusion 행렬: D = 2|s><s| - I
    diffusion_matrix = None  # TODO: 2.0 * (s_state @ s_state.conj().T) - np.eye(4, dtype=complex)
    after_diffusion = diffusion_matrix @ after_oracle if diffusion_matrix is not None and after_oracle is not None else None

    return s_state, after_oracle, after_diffusion

# [자가 검증] (수정 불필요)
s0, s1, s2 = numpy_grover_step(target='11')
assert s1 is not None and np.isclose(s1[3, 0], -0.5), 'Oracle 부호 반전 오류!'
assert s2 is not None and np.isclose(s2[3, 0], 1.0) and np.isclose(s2[0, 0], 0.0), 'Diffusion 후 진폭 증폭 오류!'
print('1. 초기 균등 중첩 |s>:\n', s0.flatten().real)
print('\n2. Oracle 적용 후 (정답 위상 반전):\n', s1.flatten().real)
print('\n3. Diffusion 적용 후 (평균 대칭 반전):\n', s2.flatten().real)
print('\n=> 오답 상태는 0으로 완전 상쇄되고, 정답 |11>만 진폭 1.0(확률 100%)으로 보강되었습니다!')


## 2. Qiskit 기반 2-qubit Grover 회로 조립

4가지 모든 타겟 상태에 대응하는 Oracle과 공통 Diffusion 회로를 조립합니다:
- **Diffusion Operator 회로**: $H^{\otimes 2} \to X^{\otimes 2} \to CZ \to X^{\otimes 2} \to H^{\otimes 2}$


In [ ]:
# [실습 과제 2] Qiskit 2-qubit Grover 탐색 회로 조립
def build_grover_circuit(target: str = '11') -> QuantumCircuit:
    qc = QuantumCircuit(2, name=f'Grover_{target}')
    
    # Step 1: 초기 균등 중첩 (|00> -> |++>)
    qc.h([0, 1])
    
    # Step 2: Phase Oracle (Little-Endian: target = 'q1 q0')
    # TODO: target 문자열에 맞춰 적절한 X 게이트와 cz(0, 1) 게이트를 배치하세요
    if target == '00':
        qc.x([0, 1]); qc.cz(0, 1); qc.x([0, 1])
    elif target == '01':  # q1=0, q0=1
        qc.x(1); qc.cz(0, 1); qc.x(1)
    elif target == '10':  # q1=1, q0=0
        qc.x(0); qc.cz(0, 1); qc.x(0)
    elif target == '11':  # q1=1, q0=1
        qc.cz(0, 1)
    
    # Step 3: Diffusion Operator (H -> X -> CZ -> X -> H)
    # TODO: 2개 큐비트에 Diffusion 패턴을 적용하세요
    qc.h([0, 1])
    qc.x([0, 1])
    qc.cz(0, 1)
    qc.x([0, 1])
    qc.h([0, 1])
    
    return qc

qc_11 = build_grover_circuit('11')
print(qc_11.draw(output='text'))


## 3. 4가지 모든 타겟에 대한 100% 탐색 검증

$|00\rangle, |01\rangle, |10\rangle, |11\rangle$ 각각에 대해 알고리즘이 정확히 100% 확률로 해당 답을 찾는지 전수 검증합니다.


In [ ]:
print('[4가지 모든 타겟에 대한 전수 검증]')
for tgt in ['00', '01', '10', '11']:
    qc = build_grover_circuit(target=tgt)
    sv = Statevector.from_instruction(qc)
    prob = sv.probabilities_dict().get(tgt, 0.0)
    print(f'  - 타겟 |{tgt}> 회로 적중 확률: {prob * 100.0:.1f}%')
    assert np.isclose(prob, 1.0)
print('=> 4개 경우 모두 단 1회의 반복으로 100% 검출됨이 증명되었습니다.')


## 4. 유한한 측정(Shots)과 통계적 요동 (Shot Noise)

이론적 확률이 100%라 하더라도 유한한 샷 수로 샘플링하면 실제 측정 카운트가 어떻게 수렴하는지 확인합니다.


In [ ]:
sv_11 = Statevector.from_instruction(qc_11)

print('[Shot 수별 타겟 |11> 측정 결과]')
for shots in [10, 50, 200, 1024, 8192]:
    counts = sv_11.sample_counts(shots=shots)
    hit_ratio = counts.get('11', 0) / shots
    print(f'  - Shots {shots:5d}: counts = {counts} -> 적중 비율 = {hit_ratio*100:.2f}%')


---
### Day 3 실습 정리 과제 (Self-Study)
1. 3개의 큐비트($N=8$)인 경우, Grover 알고리즘의 최적 반복 횟수 $R \approx \frac{\pi}{4}\sqrt{N}$를 계산해 보세요.
2. 2큐비트 탐색에서 1회가 아닌 2회를 반복하면 어떤 일이 일어나는지(과회전, Over-rotation) 수치로 확인해 보세요.
